# Model Training Workflow

This notebook demonstrates the complete model training workflow for gold price prediction.

**Requirements covered:** 5.1-5.6

## Objectives
1. Load and preprocess data
2. Engineer features
3. Split dataset into train/validation/test
4. Train multiple model architectures
5. Compare model performance
6. Save best model with metadata

In [ ]:
import sys
import os
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from src.data_ingestion import DataIngestionManager
from src.data_preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer
from src.dataset_splitter import DatasetSplitter
from src.model_training import ModelTrainingPipeline
from src.model_evaluator import ModelEvaluator
from src.model_registry import ModelRegistry
from config import Config

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Data Loading and Preprocessing

In [ ]:
# Load gold price data
data_manager = DataIngestionManager()
gold_df = data_manager.load_csv('../XAU_1d_data.csv')

print(f"Loaded {len(gold_df)} records")
print(f"Date range: {gold_df['Date'].min()} to {gold_df['Date'].max()}")

In [ ]:
# Load economic indicators
start_date = gold_df['Date'].min().strftime('%Y-%m-%d')
end_date = gold_df['Date'].max().strftime('%Y-%m-%d')
tickers = list(Config.INDICATORS.values())

print("Loading economic indicators...")
try:
    indicators = data_manager.load_economic_indicators(tickers, start_date, end_date)
    print(f"Loaded {len(indicators)} indicators")
except Exception as e:
    print(f"Warning: {e}")
    indicators = {}

In [ ]:
# Preprocess and align data
preprocessor = DataPreprocessor()

# Handle missing values and outliers
gold_df = preprocessor.handle_missing_values(gold_df)
gold_df = preprocessor.remove_outliers(gold_df)

# Align with indicators if available
if indicators:
    combined_df = preprocessor.align_datasets(gold_df, indicators)
else:
    combined_df = gold_df

print(f"Preprocessed dataset: {len(combined_df)} records")
print(f"Columns: {list(combined_df.columns)}")

## 2. Feature Engineering

In [ ]:
# Engineer features
feature_engineer = FeatureEngineer()

print("Creating lag features...")
featured_df = feature_engineer.create_lag_features(
    combined_df, 'Close', Config.LAG_PERIODS
)

print("Creating rolling features...")
featured_df = feature_engineer.create_rolling_features(
    featured_df, 'Close', Config.ROLLING_WINDOWS
)

print("Creating technical indicators...")
featured_df = feature_engineer.create_technical_indicators(featured_df)

print("Creating temporal features...")
featured_df = feature_engineer.create_temporal_features(featured_df)

if indicators:
    print("Creating interaction features...")
    featured_df = feature_engineer.create_interaction_features(featured_df)

# Drop rows with NaN (from lag/rolling operations)
featured_df = featured_df.dropna()

print(f"\nFinal feature set: {len(featured_df)} records, {len(featured_df.columns)} features")

In [ ]:
# Display sample features
print("Sample engineered features:")
print(featured_df.head())

## 3. Dataset Splitting

In [ ]:
# Split dataset
splitter = DatasetSplitter()

train_df, val_df, test_df = splitter.split_dataset(
    featured_df,
    train_ratio=Config.TRAIN_RATIO,
    val_ratio=Config.VAL_RATIO,
    test_ratio=Config.TEST_RATIO
)

print("Dataset splits:")
print(f"  Training: {len(train_df)} records ({len(train_df)/len(featured_df)*100:.1f}%)")
print(f"  Validation: {len(val_df)} records ({len(val_df)/len(featured_df)*100:.1f}%)")
print(f"  Test: {len(test_df)} records ({len(test_df)/len(featured_df)*100:.1f}%)")

# Verify integrity
is_valid = splitter.verify_split_integrity(train_df, val_df, test_df)
print(f"\nSplit integrity: {'✓ PASSED' if is_valid else '✗ FAILED'}")

In [ ]:
# Normalize features
feature_cols = [col for col in featured_df.columns if col not in ['Date']]

train_normalized, scaling_params = preprocessor.normalize_features(
    train_df[feature_cols], method=Config.NORMALIZATION_METHOD
)
val_normalized, _ = preprocessor.normalize_features(
    val_df[feature_cols], method=Config.NORMALIZATION_METHOD
)
test_normalized, _ = preprocessor.normalize_features(
    test_df[feature_cols], method=Config.NORMALIZATION_METHOD
)

print(f"Features normalized using {Config.NORMALIZATION_METHOD} method")

## 4. Prepare Data for Training

In [ ]:
# Prepare sequences for LSTM/GRU
target_idx = train_normalized.columns.get_loc('Close')

X_train_seq, y_train = splitter.create_sequences(
    train_normalized.values, 
    Config.SEQUENCE_LENGTH, 
    target_idx
)
X_val_seq, y_val = splitter.create_sequences(
    val_normalized.values,
    Config.SEQUENCE_LENGTH,
    target_idx
)
X_test_seq, y_test = splitter.create_sequences(
    test_normalized.values,
    Config.SEQUENCE_LENGTH,
    target_idx
)

print("Sequence shapes for LSTM/GRU:")
print(f"  X_train: {X_train_seq.shape}, y_train: {y_train.shape}")
print(f"  X_val: {X_val_seq.shape}, y_val: {y_val.shape}")
print(f"  X_test: {X_test_seq.shape}, y_test: {y_test.shape}")

In [ ]:
# Prepare flat features for tree-based models
X_train_flat, y_train_flat = splitter.prepare_feature_target_split(
    train_normalized, 'Close'
)
X_val_flat, y_val_flat = splitter.prepare_feature_target_split(
    val_normalized, 'Close'
)
X_test_flat, y_test_flat = splitter.prepare_feature_target_split(
    test_normalized, 'Close'
)

print("\nFlat features for XGBoost/Random Forest:")
print(f"  X_train: {X_train_flat.shape}, y_train: {y_train_flat.shape}")
print(f"  X_val: {X_val_flat.shape}, y_val: {y_val_flat.shape}")
print(f"  X_test: {X_test_flat.shape}, y_test: {y_test_flat.shape}")

## 5. Train LSTM Model

In [ ]:
# Initialize training pipeline
pipeline = ModelTrainingPipeline()

# Build LSTM model
lstm_params = {
    'units_layer1': 128,
    'units_layer2': 64,
    'dropout': 0.2,
    'learning_rate': 0.001
}

print("Building LSTM model...")
lstm_model = pipeline.build_lstm_model(X_train_seq.shape[1:], lstm_params)
print(lstm_model.summary())

In [ ]:
# Train LSTM
print("Training LSTM model...")
lstm_result = pipeline.train_model(
    lstm_model, X_train_seq, y_train, X_val_seq, y_val
)

print(f"\nTraining completed in {lstm_result.training_time:.2f} seconds")
print(f"Final training loss: {lstm_result.final_loss:.6f}")
print(f"Final validation loss: {lstm_result.validation_loss:.6f}")
print(f"Convergence status: {lstm_result.convergence_status}")

In [ ]:
# Plot training history
history = lstm_result.history

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_title('LSTM Training History - Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
if 'mae' in history:
    axes[1].plot(history['mae'], label='Training MAE', linewidth=2)
    axes[1].plot(history['val_mae'], label='Validation MAE', linewidth=2)
    axes[1].set_title('LSTM Training History - MAE', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MAE')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Train XGBoost Model

In [ ]:
# Build XGBoost model
xgb_params = {
    'max_depth': 5,
    'n_estimators': 300,
    'learning_rate': 0.05,
    'subsample': 1.0,
    'colsample_bytree': 1.0
}

print("Building XGBoost model...")
xgb_model = pipeline.build_xgboost_model(xgb_params)
print(f"XGBoost model created with params: {xgb_params}")

In [ ]:
# Train XGBoost
print("Training XGBoost model...")
xgb_result = pipeline.train_model(
    xgb_model, X_train_flat, y_train_flat, X_val_flat, y_val_flat
)

print(f"\nTraining completed in {xgb_result.training_time:.2f} seconds")
print(f"Convergence status: {xgb_result.convergence_status}")

## 7. Hyperparameter Tuning Example

In [ ]:
# Demonstrate hyperparameter tuning for XGBoost
print("Performing hyperparameter tuning (this may take several minutes)...")

# Define search space (smaller for demo)
search_space = {
    'max_depth': [3, 5],
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1]
}

try:
    best_params = pipeline.hyperparameter_tuning(
        'xgboost',
        search_space,
        X_train_flat, y_train_flat,
        X_val_flat, y_val_flat,
        method='grid'
    )
    
    print("\nBest hyperparameters found:")
    for param, value in best_params.items():
        print(f"  {param}: {value}")
except Exception as e:
    print(f"Hyperparameter tuning skipped: {e}")

## 8. Model Comparison

In [ ]:
# Evaluate both models on test set
evaluator = ModelEvaluator()

# LSTM predictions
lstm_predictions = lstm_result.model.predict(X_test_seq).flatten()
lstm_metrics = {
    'mae': evaluator.calculate_mae(y_test, lstm_predictions),
    'rmse': evaluator.calculate_rmse(y_test, lstm_predictions),
    'mape': evaluator.calculate_mape(y_test, lstm_predictions),
    'r2': evaluator.calculate_r2(y_test, lstm_predictions),
    'directional_accuracy': evaluator.calculate_directional_accuracy(y_test, lstm_predictions)
}

# XGBoost predictions
xgb_predictions = xgb_result.model.predict(X_test_flat)
xgb_metrics = {
    'mae': evaluator.calculate_mae(y_test_flat, xgb_predictions),
    'rmse': evaluator.calculate_rmse(y_test_flat, xgb_predictions),
    'mape': evaluator.calculate_mape(y_test_flat, xgb_predictions),
    'r2': evaluator.calculate_r2(y_test_flat, xgb_predictions),
    'directional_accuracy': evaluator.calculate_directional_accuracy(y_test_flat, xgb_predictions)
}

print("Model Performance Comparison on Test Set:")
print("="*60)
print(f"{'Metric':<25} {'LSTM':<15} {'XGBoost':<15}")
print("-"*60)
for metric in ['mae', 'rmse', 'mape', 'r2', 'directional_accuracy']:
    print(f"{metric.upper():<25} {lstm_metrics[metric]:<15.4f} {xgb_metrics[metric]:<15.4f}")
print("="*60)

In [ ]:
# Visualize comparison
metrics_df = pd.DataFrame({
    'LSTM': [lstm_metrics[m] for m in ['mae', 'rmse', 'mape', 'r2', 'directional_accuracy']],
    'XGBoost': [xgb_metrics[m] for m in ['mae', 'rmse', 'mape', 'r2', 'directional_accuracy']]
}, index=['MAE', 'RMSE', 'MAPE', 'R²', 'Directional Accuracy'])

fig, ax = plt.subplots(figsize=(10, 6))
metrics_df.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xlabel('Metric')
ax.set_ylabel('Value')
ax.legend(title='Model')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 9. Save Best Model

In [ ]:
# Determine best model based on RMSE
best_model_name = 'LSTM' if lstm_metrics['rmse'] < xgb_metrics['rmse'] else 'XGBoost'
best_model = lstm_result.model if best_model_name == 'LSTM' else xgb_result.model
best_metrics = lstm_metrics if best_model_name == 'LSTM' else xgb_metrics

print(f"Best model: {best_model_name} (RMSE: {best_metrics['rmse']:.4f})")

In [ ]:
# Save model with metadata
from src.model_registry import ModelMetadata

version = f"notebook_v{datetime.now().strftime('%Y%m%d_%H%M%S')}"

metadata = ModelMetadata(
    version=version,
    model_type=best_model_name,
    training_date=datetime.now(),
    hyperparameters=lstm_params if best_model_name == 'LSTM' else xgb_params,
    feature_list=list(train_normalized.columns),
    scaling_params=scaling_params,
    performance_metrics=best_metrics,
    training_data_range=(train_df['Date'].min(), train_df['Date'].max()),
    sequence_length=Config.SEQUENCE_LENGTH if best_model_name == 'LSTM' else None
)

model_path = pipeline.save_model(best_model, metadata, version)
print(f"Model saved to: {model_path}")

In [ ]:
# Register model
registry = ModelRegistry()
registry.register_model(model_path, metadata, version)
print(f"Model registered with version: {version}")

## Summary

This notebook demonstrated:
1. ✅ Complete data preprocessing pipeline
2. ✅ Feature engineering with lag, rolling, and technical indicators
3. ✅ Dataset splitting for time series
4. ✅ Training LSTM and XGBoost models
5. ✅ Hyperparameter tuning example
6. ✅ Model comparison and evaluation
7. ✅ Model persistence and versioning

**Key Results:**
- Best Model: {best_model_name}
- Test RMSE: {best_metrics['rmse']:.4f}
- Test R²: {best_metrics['r2']:.4f}
- Directional Accuracy: {best_metrics['directional_accuracy']:.2%}

**Next Steps:**
- Proceed to `03_prediction_and_forecasting.ipynb` to use the trained model
- Experiment with different hyperparameters
- Try other model architectures (GRU, Random Forest)